In [2]:
import sys 

sys.path.append('/home/amos/programs/CineFace')

import ast 
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm 
from qdrant_client import QdrantClient 

from metadata import get_cast

In [4]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)

In [5]:
df = pd.read_csv('../data/label_df.csv', index_col=0)
df.head()

,x1,y1,x2,y2,encoding,tmdb_id,filepath,series_id,season,episode,pct_of_frame,frame_num,face_num
0,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0
1,726,204,1140,762,"[-1.8272753953933716, -1.2740103006362915, 0.3...",15854.0,/home/amos/media/tv/Monk.(2002).Season.1-8.S01...,312172,1,7,0.111406,30293.0,0.0
2,1242,324,1404,522,"[0.4579177796840668, 0.17056649923324585, -1.0...",132319.0,/home/amos/media/tv/Frasier Complete S01-S11 b...,106004,11,12,0.015469,30654.0,0.0
3,1278,132,1548,486,"[-0.12152212858200073, 0.24677366018295288, 0....",6194.0,/home/amos/media/tv/homeland/Homeland.S06.1080...,1796960,6,1,0.046094,8096.0,0.0
4,1047,202,1214,417,"[0.3512590825557709, -1.311365008354187, -1.24...",10872.0,/home/amos/media/tv/The.King.of.Queens.S01-S09...,165581,4,10,0.017351,11185.0,0.0


In [30]:
def validate(row, r):
    tmdb_id = int(row['tmdb_id']) if not pd.isnull(row['tmdb_id']) else None
    try:
        if pd.isnull(tmdb_id) and not r:
            status = 'tn'
        elif not pd.isnull(tmdb_id) and not r:
            status = 'fn'
        elif pd.isnull(tmdb_id) and r:
            status = 'fp'
        elif not pd.isnull(tmdb_id) and r and r[0].payload['tmdb_id'] == row['tmdb_id']:
            status = 'tp' 
        elif not pd.isnull(tmdb_id) and r and r[0].payload['tmdb_id'] != row['tmdb_id']:
            status = 'mp'
        return status
    except:
        print(r, row)
        return None

In [ ]:
data = []
sample = pd.read_csv('/home/amos/programs/CineFace/research/data/label_df.csv', index_col=0)
for idx, row in tqdm(sample.iterrows(), total=sample.shape[0]):
    imdb_id = row['series_id']
    season = row['season']
    episode = row['episode']

    cast = get_cast(imdb_id, season, episode)
    cast_ids = [x['id'] for x in cast]
                            
    encoding = ast.literal_eval(row['encoding'])

    error = False
    for threshold in np.arange(0, 1, 0.05):
        response = CLIENT.query_points(collection_name='Headshots',
                                        with_payload=True,
                                        query=encoding,
                                        limit=100)
        r = [x for x in response.points if x.payload['tmdb_id'] in cast_ids and x.score > threshold]
        status = validate(row, r)
        if not status:
            error = True
            break 

        datum = {**row.to_dict(), 'status': status, 'threshold': threshold}
        data.append(datum)    
    if error:
        break
result_df = pd.DataFrame(data)


  0%|          | 0/385 [00:00<?, ?it/s]

100%|██████████| 385/385 [03:40<00:00,  1.75it/s]


In [32]:
result_df.head()

,x1,y1,x2,y2,encoding,tmdb_id,filepath,series_id,season,episode,pct_of_frame,frame_num,face_num,status,threshold
0,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,tn,0.00
1,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,tn,0.05
2,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,tn,0.10
3,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,tn,0.15
4,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,tn,0.20


In [33]:
temp = result_df[result_df['threshold'] == 0.5]
g = temp.groupby('status').size()
g

status
fn    152
fp      1
mp      5
tn     13
tp    214
dtype: int64

In [41]:
(g['tp'] + g['fp']) / g.values.sum()

0.5584415584415584

In [40]:
g.values.sum()

385

In [15]:
df = pd.read_csv('../data/label_df_results.csv', index_col=0)
df.head()

,x1,y1,x2,y2,encoding,tmdb_id,filepath,series_id,season,episode,pct_of_frame,frame_num,face_num,detector,recognition_model,status,threshold
0,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,opencv,VGG-Face,fp,0.00
1,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,opencv,VGG-Face,fp,0.05
2,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,opencv,VGG-Face,fp,0.10
3,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,opencv,VGG-Face,fp,0.15
4,714,144,900,402,"[0.20887960493564606, -0.05747608467936516, -1...",NaN,/home/amos/media/tv/Raising Hope (2010) Season...,1615919,3,14,0.023142,10591.0,0.0,opencv,VGG-Face,tn,0.20


In [20]:
filenames = df['filepath'].unique().tolist()

In [24]:
df[df['filepath'] == filenames[3]]

IndexError: list index out of range

In [26]:
filenames

['/home/amos/media/tv/Raising Hope (2010) Season 1-4 S01-S04 (1080p AMZN WEB-DL x265 HEVC 10bit EAC3 5.1 RZeroX)/Season 3/Raising Hope (2010) - S03E14 - Modern Wedding (1080p AMZN WEB-DL x265 RZeroX).mkv.mkv',
 '/home/amos/media/tv/Monk.(2002).Season.1-8.S01-S08.(1080p.AMZN.WEB-DL.x265.HEVC.10bit.EAC3.Mixed.RZeroX)/Season 1/Monk (2002) - S01E07 - Mr. Monk and the Billionaire Mugger (1080p AMZN WEB-DL x265 RZeroX).mkv',
 '/home/amos/media/tv/Frasier Complete S01-S11 br 10bit dts hevc-d3g/Fraiser S11 br 10bit dts hevc-d3g/Frasier.S11E12.Frasier.Lite.1080p.BluRay.10Bit.Dts-HDMa2.0.HEVC-d3g.mkv']